# OpenInterp — Dev Session (pre-PyPI)

Use this notebook to test openinterp-mcp **before the PyPI release**. It installs the package directly from a local upload or from a GitHub branch.

Path A: upload the `openinterp_mcp/` source as a zip to this Colab session.
Path B: install from a public GitHub branch (once you've pushed).

Pick one cell below (A or B), then run the remaining cells.

## Setup secrets

Add these to Colab Secrets (🔑 left sidebar):
- `NGROK_AUTHTOKEN` (required) — https://dashboard.ngrok.com
- `HF_TOKEN` (optional)
- `ANTHROPIC_API_KEY` (optional — only for judge tools)

In [ ]:
from google.colab import userdata
import os
for k in ['NGROK_AUTHTOKEN', 'HF_TOKEN', 'ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass
print('Secrets loaded:', [k for k in ('NGROK_AUTHTOKEN','HF_TOKEN','ANTHROPIC_API_KEY','OPENAI_API_KEY') if os.environ.get(k)])

## Path A — install from local zip upload

1. On your laptop, in the openinterp-mcp repo root:
   ```bash
   zip -r openinterp_mcp.zip openinterp_mcp pyproject.toml README.md LICENSE
   ```
2. In Colab → Files panel → upload `openinterp_mcp.zip`.
3. Run the cell below.

In [ ]:
# Path A: install from uploaded zip
!unzip -o openinterp_mcp.zip -d /content/openinterp-mcp-src
%pip install -q /content/openinterp-mcp-src
%pip install -q transformers torch safetensors scikit-learn pyngrok fastapi uvicorn

## Path B — install from GitHub branch

Skip if you used Path A. Replace the branch name as needed.

In [ ]:
# Path B: install from GitHub
# %pip install -q 'git+https://github.com/OpenInterpretability/openinterp-mcp.git@main#egg=openinterp-mcp[colab]'

## Launch the backend

Loads Qwen2.5-7B-Instruct (fits on Colab Pro A100; on Free T4 use a smaller model like Qwen2.5-1.5B-Instruct or Qwen2.5-3B-Instruct).

In [ ]:
from openinterp_mcp.colab import launch

url = launch(
    model='Qwen/Qwen2.5-3B-Instruct',  # change to 7B if you have A100
    device='cuda',
    dtype='bfloat16',
)

## Quick verification (run inside Colab)

Confirms the backend responds before you paste the URL into Claude.

In [ ]:
import httpx
print('/health →', httpx.get(f'{url}/health').json())
print()
cap = httpx.post(f'{url}/capture', json={
    'prompt': 'Solve x^2 = 4',
    'layers': [10, 14, 20],
    'positions': ['end_question']
}, timeout=120).json()
print('/capture →', cap)

## Paste the URL where the agent can see it

Two ways to connect:

**Via MCP server** (after you've installed `openinterp-mcp[server]` on your laptop and registered in `claude_desktop_config.json` / `~/.cursor/mcp.json`):

```
/colab-attach https://abc.ngrok-free.app
```

**Via direct curl** (no MCP install — works today, agent uses Bash):

Just paste the URL in your message. The agent will hit `{url}/capture`, `{url}/probe`, etc. via curl.